# Moving object detection on Zynq (GMM + single-pass CCL)
Copy `mod_overlay.bit`, `mod_overlay.hwh`, `mod_driver.py`, `mod_model.py` and a test video into this folder.

In [ ]:
import cv2, numpy as np, time
from IPython.display import Image, display
from mod_driver import MovingObjectDetector, draw_objects

W, H = 640, 480
det = MovingObjectDetector('mod_overlay.bit', W, H, min_area=64)
print('max width', det.max_w, 'max objects', det.max_obj)

In [ ]:
cap = cv2.VideoCapture('vtest.avi')   # or cv2.VideoCapture(0) for a USB camera
def next_frame():
    ok, fr = cap.read()
    if not ok:
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0); ok, fr = cap.read()
    return cv2.resize(fr, (W, H), interpolation=cv2.INTER_AREA)

In [ ]:
# learn the background for a few frames, then show one result
for i in range(60):
    mask, objs, st = det.process(next_frame())
fr = next_frame()
mask, objs, st = det.process(fr)
vis = np.hstack([draw_objects(fr, objs), cv2.cvtColor(np.array(mask), cv2.COLOR_GRAY2BGR)])
display(Image(data=cv2.imencode('.jpg', vis)[1].tobytes()))
print(len(objs), 'objects', st)

In [ ]:
# throughput
t_hw, t_all = [], []
for i in range(200):
    fr = next_frame()
    t0 = time.perf_counter()
    mask, objs, st = det.process(fr)
    t_all.append(time.perf_counter() - t0); t_hw.append(st['t_hw'])
print('PL + DMA   : %.2f ms  (%.1f fps)' % (1e3*np.mean(t_hw), 1/np.mean(t_hw)))
print('with Python: %.2f ms  (%.1f fps)' % (1e3*np.mean(t_all), 1/np.mean(t_all)))

In [ ]:
# software baseline on the ARM cores (OpenCV MOG2 + closing/opening + CCL)
mog = cv2.createBackgroundSubtractorMOG2(history=128, varThreshold=6.25, detectShadows=False)
mog.setNMixtures(3)
k = np.ones((3,3), np.uint8); ts = []
for i in range(100):
    fr = next_frame(); t0 = time.perf_counter()
    m = cv2.morphologyEx(cv2.morphologyEx(mog.apply(fr, learningRate=1/128), cv2.MORPH_CLOSE, k), cv2.MORPH_OPEN, k)
    cv2.connectedComponentsWithStats(m, connectivity=8)
    ts.append(time.perf_counter() - t0)
print('ARM software: %.2f ms (%.1f fps)' % (1e3*np.mean(ts[5:]), 1/np.mean(ts[5:])))

In [ ]:
det.close()